In [1]:
import torch, gc

gc.collect()
torch.cuda.empty_cache()

print("GPU available:", torch.cuda.is_available())

GPU available: True


In [2]:
# ============================================================
# FULL PIPELINE: FinBERT + LSTM for Stock Movement Prediction
# Optimized for Kaggle GPU + FNSPID (nasdaq_external_data.csv)
# ============================================================

# ─────────────────────────────────────────────
# STEP 0: Install & Imports
# ─────────────────────────────────────────────
!pip install transformers torch tqdm scikit-learn --quiet

import os
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")


In [3]:

# ─────────────────────────────────────────────
# STEP 1: Config
# ─────────────────────────────────────────────
FILE_PATH    = "/kaggle/input/datasets/ramavaibhav/nasdaq-external-data-csv/nasdaq_exteral_data.csv"
PRICE_FOLDER = "/kaggle/input/full-history/full_history"
OUTPUT_SENT  = "/kaggle/working/sentiment_output.csv"
OUTPUT_FINAL = "/kaggle/working/final_merged.csv"

CHUNKSIZE  = 20000
BATCH_SIZE = 32          # FinBERT batch size per GPU pass
SEQ_LEN    = 60          # 60-day lookback window
EPOCHS     = 30
LR         = 0.001
MAX_ROWS   = None        # Set to e.g. 500_000 to debug faster; None = full dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:

# ─────────────────────────────────────────────
# STEP 2: Load FinBERT
# ─────────────────────────────────────────────
print("\n[1/7] Loading FinBERT...")

# If internet is ON:
FINBERT_PATH = "ProsusAI/finbert"

# If internet is OFF (offline Kaggle dataset):
# FINBERT_PATH = "/kaggle/input/finbert"

tokenizer = AutoTokenizer.from_pretrained(FINBERT_PATH)
finbert   = AutoModelForSequenceClassification.from_pretrained(FINBERT_PATH)
finbert.to(device)
finbert.eval()
print("FinBERT loaded.")


[1/7] Loading FinBERT...


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

FinBERT loaded.


In [ ]:
# ─────────────────────────────────────────────
# STEP 3: FinBERT Sentiment Extraction
#          (chunk + GPU batch processing)
# ─────────────────────────────────────────────
def get_finbert_sentiment(texts, batch_size=BATCH_SIZE):
    """
    Runs FinBERT on a list of texts.
    Returns numpy array of shape (N, 3): [pos, neg, neu]
    FinBERT label order: positive=0, negative=1, neutral=2
    """
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        # Replace NaN / empty strings
        batch = [str(t) if pd.notna(t) and t != "" else "neutral" for t in batch]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            logits = finbert(**encoded).logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        all_probs.append(probs)
    return np.vstack(all_probs)


print("\n[2/7] Extracting sentiment from news (chunk processing)...")
sentiment_rows = []
rows_read = 0

for chunk in pd.read_csv(
        FILE_PATH,
        chunksize=CHUNKSIZE,
        usecols=["Date", "Stock_symbol", "Article_title", "Article"],
        on_bad_lines="skip",
        low_memory=False):

    # ── Debug cap ──
    if MAX_ROWS and rows_read >= MAX_ROWS:
        break
    rows_read += len(chunk)

    # ── Combine title + body for richer context ──
    texts = (
        chunk["Article_title"].fillna("") + " " +
        chunk["Article"].fillna("")
    ).tolist()

    probs = get_finbert_sentiment(texts)

    chunk["pos"] = probs[:, 0]
    chunk["neg"] = probs[:, 1]
    chunk["neu"] = probs[:, 2]

    sentiment_rows.append(
        chunk[["Date", "Stock_symbol", "pos", "neg", "neu"]]
    )

    # Free GPU memory every 10 chunks
    if len(sentiment_rows) % 10 == 0:
        torch.cuda.empty_cache()
        gc.collect()
        print(f"  Processed {rows_read:,} rows...")

# ── Save raw sentiment ──
sentiment_df = pd.concat(sentiment_rows, ignore_index=True)
sentiment_df["Date"] = pd.to_datetime(sentiment_df["Date"], errors="coerce")
sentiment_df.dropna(subset=["Date", "Stock_symbol"], inplace=True)
sentiment_df.to_csv(OUTPUT_SENT, index=False)
print(f"Sentiment saved → {OUTPUT_SENT}  Shape: {sentiment_df.shape}")

# Free FinBERT from GPU (no longer needed)
del finbert
torch.cuda.empty_cache()
gc.collect()


[2/7] Extracting sentiment from news (chunk processing)...


In [ ]:

# ─────────────────────────────────────────────
# STEP 4: Aggregate to Daily Sentiment
# ─────────────────────────────────────────────
print("\n[3/7] Aggregating daily sentiment...")

daily_sentiment = (
    sentiment_df
    .groupby(["Stock_symbol", "Date"])[["pos", "neg", "neu"]]
    .mean()
    .reset_index()
)

# Derived signal: sentiment score in [-1, +1]
daily_sentiment["sentiment_score"] = (
    daily_sentiment["pos"] - daily_sentiment["neg"]
)

print(f"Daily sentiment shape: {daily_sentiment.shape}")


In [ ]:
# ─────────────────────────────────────────────
# STEP 5: Load Stock Price Data
# ─────────────────────────────────────────────
print("\n[4/7] Loading stock prices...")

price_chunks = []

for fname in os.listdir(PRICE_FOLDER):
    if not fname.endswith(".csv"):
        continue
    fpath  = os.path.join(PRICE_FOLDER, fname)
    ticker = fname.replace(".csv", "").upper()
    try:
        df = pd.read_csv(fpath, low_memory=False)
        # Normalise column names (different files may differ in case)
        df.columns = df.columns.str.strip().str.lower()
        df.rename(columns={"date": "Date", "close": "Close",
                            "volume": "Volume", "open": "Open",
                            "high": "High", "low": "Low"}, inplace=True)
        df["Stock_symbol"] = ticker
        keep = [c for c in ["Date","Open","High","Low","Close","Volume","Stock_symbol"]
                if c in df.columns]
        price_chunks.append(df[keep])
    except Exception as e:
        print(f"  Skipping {fname}: {e}")

prices = pd.concat(price_chunks, ignore_index=True)
prices["Date"] = pd.to_datetime(prices["Date"], errors="coerce")
prices.dropna(subset=["Date", "Close"], inplace=True)
prices.sort_values(["Stock_symbol", "Date"], inplace=True)
print(f"Prices shape: {prices.shape}")


In [ ]:

# ─────────────────────────────────────────────
# STEP 6: Merge Price + Sentiment
# ─────────────────────────────────────────────
print("\n[5/7] Merging price + sentiment...")

data = pd.merge(prices, daily_sentiment, on=["Stock_symbol", "Date"], how="inner")
data.sort_values(["Stock_symbol", "Date"], inplace=True)
data.to_csv(OUTPUT_FINAL, index=False)
print(f"Merged dataset shape: {data.shape}")


In [ ]:
# ─────────────────────────────────────────────
# STEP 7: Build t-60 Sequences
# ─────────────────────────────────────────────
print("\n[6/7] Building sequences (SEQ_LEN={SEQ_LEN})...")

FEATURE_COLS = ["Close", "Volume", "pos", "neg", "neu", "sentiment_score"]

X_list, y_list = [], []
scaler_map = {}   # store per-ticker scaler for inverse transform later

for ticker, grp in data.groupby("Stock_symbol"):
    grp = grp.dropna(subset=FEATURE_COLS).reset_index(drop=True)

    if len(grp) < SEQ_LEN + 1:
        continue

    # ── Scale features per ticker ──
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(grp[FEATURE_COLS].values)
    scaler_map[ticker] = scaler

    for i in range(SEQ_LEN, len(scaled)):
        X_list.append(scaled[i - SEQ_LEN : i])       # shape (60, n_features)
        y_list.append(scaled[i, 0])                   # next-day Close (index 0)

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.float32)

print(f"X shape: {X.shape}  |  y shape: {y.shape}")

# ─────────────────────────────────────────────
# Train / Val / Test Split  (70 / 15 / 15)
# ─────────────────────────────────────────────
n       = len(X)
n_train = int(n * 0.70)
n_val   = int(n * 0.85)

X_train, y_train = X[:n_train],   y[:n_train]
X_val,   y_val   = X[n_train:n_val], y[n_train:n_val]
X_test,  y_test  = X[n_val:],     y[n_val:]

def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32).to(device)

train_loader = DataLoader(
    TensorDataset(to_tensor(X_train), to_tensor(y_train)),
    batch_size=256, shuffle=True
)
val_loader = DataLoader(
    TensorDataset(to_tensor(X_val), to_tensor(y_val)),
    batch_size=256
)


In [ ]:
# ─────────────────────────────────────────────
# STEP 8: LSTM Model
# ─────────────────────────────────────────────
class LSTMPredictor(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)         # (batch, seq, hidden)
        out    = self.dropout(out[:, -1, :])   # last timestep
        return self.fc(out).squeeze(1)

input_size = X.shape[2]              # number of features
lstm_model = LSTMPredictor(input_size).to(device)
print(f"\nModel parameters: {sum(p.numel() for p in lstm_model.parameters()):,}")


In [ ]:
# ─────────────────────────────────────────────
# STEP 9: Training
# ─────────────────────────────────────────────
print("\n[7/7] Training LSTM...")

optimizer  = torch.optim.Adam(lstm_model.parameters(), lr=LR)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode="min", patience=3, factor=0.5, verbose=True)
loss_fn    = nn.MSELoss()

best_val_loss = float("inf")
patience_cnt  = 0
EARLY_STOP    = 7

for epoch in range(1, EPOCHS + 1):

    # ── Train ──
    lstm_model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        pred = lstm_model(xb)
        loss = loss_fn(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(lstm_model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item() * len(xb)
    train_loss /= len(train_loader.dataset)

    # ── Validate ──
    lstm_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            pred = lstm_model(xb)
            val_loss += loss_fn(pred, yb).item() * len(xb)
    val_loss /= len(val_loader.dataset)

    scheduler.step(val_loss)
    print(f"Epoch {epoch:3d} | Train MSE: {train_loss:.6f} | Val MSE: {val_loss:.6f}")

    # ── Early Stopping ──
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_cnt  = 0
        torch.save(lstm_model.state_dict(), "/kaggle/working/best_lstm.pt")
    else:
        patience_cnt += 1
        if patience_cnt >= EARLY_STOP:
            print(f"Early stopping at epoch {epoch}")
            break


In [ ]:
# ─────────────────────────────────────────────
# STEP 10: Evaluation on Test Set
# ─────────────────────────────────────────────
print("\n── Test Evaluation ──")

lstm_model.load_state_dict(torch.load("/kaggle/working/best_lstm.pt"))
lstm_model.eval()

X_test_t = to_tensor(X_test)
with torch.no_grad():
    preds = lstm_model(X_test_t).cpu().numpy()

mae  = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print(f"Test MAE  (scaled): {mae:.6f}")
print(f"Test RMSE (scaled): {rmse:.6f}")


In [ ]:
# ─────────────────────────────────────────────
# STEP 11: Save Predictions
# ─────────────────────────────────────────────
results_df = pd.DataFrame({
    "actual":    y_test,
    "predicted": preds
})
results_df.to_csv("/kaggle/working/predictions.csv", index=False)
print("\nPredictions saved → /kaggle/working/predictions.csv")
print("Best model saved  → /kaggle/working/best_lstm.pt")
print("\nDone ✓")